In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import glob

# ==========================================
# 自动加载并拼接数据 (Cell 1 中运行过可省略)
# ==========================================
base_path = r'D:\Workspace\5_Data\bupt_ring_selftest\wsx\报告\green\parsed_results'
search_pattern = os.path.join(base_path, '202*.csv') 
file_list = sorted(glob.glob(search_pattern))

if file_list:
    df_list = [pd.read_csv(f) for f in file_list]
    df = pd.concat(df_list, ignore_index=True)
    print(f"✅ 数据加载就绪，总行数: {len(df)}")

# ==========================================
# 交互式可视化模块 (Cell 2)
# ==========================================
if 'green1' not in df.columns or 'butter_green1' not in df.columns:
    print("⚠️ 警告：数据中未找到目标列，请确保已运行新的 Numpy 滤波脚本！")
else:
    max_len = len(df)
    
    start_slider = widgets.IntSlider(
        min=0, max=max_len-100, step=100, value=0, 
        description='开始索引:', 
        layout=widgets.Layout(width='90%'),
        continuous_update=False 
    )
    
    window_slider = widgets.IntSlider(
        min=100, max=10000, step=100, value=1000, 
        description='窗口宽度:', 
        layout=widgets.Layout(width='90%'),
        continuous_update=False
    )
    
    out = widgets.Output()

    def update_plot(*args):
        start = start_slider.value
        window = window_slider.value
        end = min(start + window, max_len)
        
        if start >= max_len: return
        df_slice = df.iloc[start:end].copy()
        
        # 修复 100Hz 时间轴
        if 'Date' in df_slice.columns and 'Time' in df_slice.columns:
            first_dt_str = str(df_slice['Date'].iloc[0]) + ' ' + str(df_slice['Time'].iloc[0])
            base_time = pd.to_datetime(first_dt_str, dayfirst=True, errors='coerce')
            t_axis = base_time + pd.to_timedelta(np.arange(len(df_slice)) * 10, unit='ms')
            xlabel = "Time (100Hz Interpolated)"
        else:
            t_axis = np.arange(start, end)
            xlabel = "Data Index"
        
        y_raw = df_slice['green1'].values
        y_filtered = df_slice['butter_green1'].values

        with out:
            clear_output(wait=True) 
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
            
            # 子图 1：原始数据
            ax1.plot(t_axis, y_raw, color='#2ca02c', linewidth=1.2, label='Raw green1')
            ax1.set_title(f'Raw PPG Data (Index {start} to {end})', fontweight='bold')
            ax1.legend(loc='upper right')
            ax1.grid(True, linestyle='--', alpha=0.5)
            
            # 子图 2：FFT 滤波后数据
            ax2.plot(t_axis, y_filtered, color='#1f77b4', linewidth=1.5, label='Numpy FFT Filtered (0.5-8Hz)')
            ax2.set_title('FFT Filtered PPG Data (Zero Phase Shift)', fontweight='bold')
            ax2.set_xlabel(xlabel)
            ax2.legend(loc='upper right')
            ax2.grid(True, linestyle='--', alpha=0.5)
            
            plt.tight_layout()
            plt.show()      
            plt.close(fig)  

    start_slider.observe(update_plot, 'value')
    window_slider.observe(update_plot, 'value')

    ui = widgets.VBox([
        widgets.HTML("<h3>🎛️ 绕过拦截的极速 PPG 浏览器</h3>"),
        start_slider, window_slider, out
    ])
    
    display(ui)
    update_plot()

✅ 数据加载就绪，总行数: 8262961


In [2]:
df.columns

Index(['Date', 'Time', 'Duration', 'green1', 'green2', 'ir1', 'ir2', 'accX',
       'accY', 'accZ', 'clean_green1', 'butter_green1'],
      dtype='str')

In [3]:
import os
import numpy as np
import pandas as pd

base_path = r'D:\Workspace\5_Data\bupt_ring_selftest\wsx\报告\green\parsed_results'

# 1. 读取保存好的心搏特征表
features_path = os.path.join(base_path, 'ppg_extracted_features.csv')
features_df = pd.read_csv(features_path)

# 2. 读取保存好的连续交流波形矩阵
ac_signal_path = os.path.join(base_path, 'ac_signal.npy')
ac_signal = np.load(ac_signal_path)

print(f"✅ 结果秒加载完成！")
print(f"📊 特征数量: {len(features_df)} 个心跳")
print(f"📈 波形长度: {len(ac_signal)} 个采样点")

✅ 结果秒加载完成！
📊 特征数量: 90006 个心跳
📈 波形长度: 8262961 个采样点


In [7]:
features_df.dtypes

datetime            str
rri_raw         float64
hr_raw          float64
hr_display        int64
area_up           int64
area_down         int64
motion            int64
peak_idx          int64
valley_idx        int64
is_recovered      int64
dtype: object

In [5]:
# 1. 声明模式
%matplotlib inline 

import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from IPython.display import clear_output
import numpy as np

# ================= 核心优化点 1：把重度计算移出滑动循环 =================
# 提前将 DataFrame 的索引提取为 Numpy 数组，Numpy 的切片过滤比 Pandas 快上百倍
# 假设你的 features_df 和 ac_signal 已经准备好
all_peaks = features_df['peak_idx'].astype(int).values
all_valleys = features_df['valley_idx'].astype(int).values
total_points = len(ac_signal)

def interactive_ppg_plot(start_idx, window_size):
    # ================= 核心优化点 2：使用 clear_output 避免前端 DOM 堆叠 =================
    clear_output(wait=True) 
    
    end_idx = min(start_idx + window_size, total_points)
    time_axis = np.arange(start_idx, end_idx)
    ac_slice = ac_signal[start_idx:end_idx]
    
    # 使用 Numpy 布尔掩码进行极速过滤
    window_peaks = all_peaks[(all_peaks >= start_idx) & (all_peaks < end_idx)]
    window_valleys = all_valleys[(all_valleys >= start_idx) & (all_valleys < end_idx)]
    
    fig, ax = plt.subplots(figsize=(15, 5))
    
    # 绘制基础交流波形
    ax.plot(time_axis, ac_slice, color='#2ca02c', linewidth=1.5, label='AC Signal')
    
    # 标注波峰和波谷
    if len(window_peaks) > 0:
        ax.scatter(window_peaks, ac_signal[window_peaks], 
                   color='red', s=80, zorder=5, label='Peaks')
    if len(window_valleys) > 0:
        ax.scatter(window_valleys, ac_signal[window_valleys], 
                   color='blue', s=80, zorder=5, label='Valleys')
    
    # 图表装饰
    ax.set_title(f'PPG Signal Visualization (Index: {start_idx} to {end_idx})', fontsize=14)
    ax.set_xlabel('Sample Index (100Hz = 10ms/point)', fontsize=12)
    ax.set_ylabel('Amplitude', fontsize=12)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1) 
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 显式关闭当前画布，配合 clear_output 使用更安全
    plt.close(fig) 

# ================= 触发交互组件 =================
max_start = max(0, total_points - 500) 

interact(
    interactive_ppg_plot, 
    start_idx=IntSlider(min=0, max=max_start, step=200, value=0, description='起点索引:', continuous_update=False),
    window_size=IntSlider(min=200, max=3000, step=100, value=1000, description='窗口大小:', continuous_update=False)
);

interactive(children=(IntSlider(value=0, continuous_update=False, description='起点索引:', max=8262461, step=200),…

In [10]:
# 1. 声明模式
%matplotlib inline 

import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from IPython.display import clear_output
import numpy as np

# ================= 核心优化点 1：把重度计算移出滑动循环 =================
# 假设你的 features_df 和 ac_signal 已经准备好
# ⚠️ 注意：你需要从最原始的 DataFrame 中提取 raw_signal，请替换 'green1' 为你的实际列名
raw_signal = -1. * df['green1'].values  

all_peaks = features_df['peak_idx'].astype(int).values
all_valleys = features_df['valley_idx'].astype(int).values
all_motions = features_df['motion'].values  # 提取运动量特征
total_points = len(ac_signal)

def interactive_ppg_plot(start_idx, window_size):
    # ================= 核心优化点 2：使用 clear_output 避免前端 DOM 堆叠 =================
    clear_output(wait=True) 
    
    end_idx = min(start_idx + window_size, total_points)
    time_axis = np.arange(start_idx, end_idx)
    
    # 连续信号切片
    ac_slice = ac_signal[start_idx:end_idx]
    raw_slice = raw_signal[start_idx:end_idx]
    
    # 使用 Numpy 布尔掩码进行极速过滤
    # 获取当前窗口内的特征点索引掩码
    peak_mask = (all_peaks >= start_idx) & (all_peaks < end_idx)
    window_peaks = all_peaks[peak_mask]
    window_motions = all_motions[peak_mask]  # 同步获取对应的 motion
    
    valley_mask = (all_valleys >= start_idx) & (all_valleys < end_idx)
    window_valleys = all_valleys[valley_mask]
    
    # ================= 创建 3 行 1 列的共享 X 轴子图 =================
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
    
    # -------- 子图 1: 原始 PPG (含基线漂移) --------
    ax1.plot(time_axis, raw_slice, color='gray', linewidth=1.5, label='Raw PPG Signal')
    ax1.set_title(f'Multi-Modal PPG Visualization (Index: {start_idx} to {end_idx})', fontsize=14, pad=10)
    ax1.set_ylabel('Raw Amplitude', fontsize=12)
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # -------- 子图 2: 交流信号与波峰波谷 --------
    ax2.plot(time_axis, ac_slice, color='#2ca02c', linewidth=1.5, label='AC Signal')
    if len(window_peaks) > 0:
        ax2.scatter(window_peaks, ac_signal[window_peaks], 
                    color='red', s=80, zorder=5, label='Peaks')
    if len(window_valleys) > 0:
        ax2.scatter(window_valleys, ac_signal[window_valleys], 
                    color='blue', s=80, zorder=5, label='Valleys')
    ax2.set_ylabel('AC Amplitude', fontsize=12)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1) 
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)
    
    # -------- 子图 3: Motion 运动量 --------
    if len(window_peaks) > 0:
        # 因为 motion 是基于心跳周期的离散特征，我们用阶梯线或连线画在对应的波峰 x 轴上
        ax3.plot(window_peaks, window_motions, color='darkorange', marker='o', 
                 linestyle='-', linewidth=2, label='Motion (Mean ENMO)')
        # 填充运动量下方的面积，视觉更直观
        ax3.fill_between(window_peaks, window_motions, 0, color='darkorange', alpha=0.2)
        
    ax3.set_xlabel('Sample Index (100Hz = 10ms/point)', fontsize=12)
    ax3.set_ylabel('Motion Level', fontsize=12)
    ax3.set_ylim((0, 500))  # 运动量通常大于0，锁定底部
    ax3.legend(loc='upper right')
    ax3.grid(True, alpha=0.3)
    
    # 自动调整子图间距
    plt.tight_layout()
    plt.show()
    
    # 显式关闭当前画布，配合 clear_output 使用更安全
    plt.close(fig) 

# ================= 触发交互组件 =================
max_start = max(0, total_points - 500) 

interact(
    interactive_ppg_plot, 
    start_idx=IntSlider(min=0, max=max_start, step=200, value=0, description='起点索引:', continuous_update=False),
    window_size=IntSlider(min=200, max=3000, step=100, value=1000, description='窗口大小:', continuous_update=False)
);

interactive(children=(IntSlider(value=0, continuous_update=False, description='起点索引:', max=8262461, step=200),…